#### npz File LogMel


In [2]:
import os
import librosa
import numpy as np

ulazni_dir = r'C:\PETNICA\RAC\RAC2.1\LetnjiPripreme\datasetSpeech_1'
fajlovi = [f for f in os.listdir(ulazni_dir) if f.endswith(".wav")]

EMOTION_MAP = {
    "01": 0,  # neutral
    "02": 1,  # calm
    "03": 2,  # happy
    "04": 3,  # sad
    "05": 4,  # angry
    "06": 5,  # fearful
    "07": 6,  # disgust
    "08": 7,  # surprised
}

X = []
y = []

target_sr = 16000
n_mels = 128

print(f"Započeta ekstrakcija Mel-spektrograma za {len(fajlovi)} fajlova...")

for i, fajl in enumerate(fajlovi):
    try:
        deli = fajl.split("-")
        emocija_kod = deli[2]
        labela = EMOTION_MAP[emocija_kod]

        
        putanja = os.path.join(ulazni_dir, fajl)
        audio, sr = librosa.load(putanja, sr=target_sr, mono=True)

        
        mel_spec = librosa.feature.melspectrogram(
            y=audio, sr=sr, n_mels=n_mels, fmax=8000
        )
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        X.append(mel_spec_db)
        y.append(labela)

    except Exception as e:
        print(f"Greška na fajlu {fajl}: {e}")

    if (i + 1) % 200 == 0 or (i + 1) == len(fajlovi):
        print(f"Obrađeno {i + 1}/{len(fajlovi)}...")


X = np.array(X)
y = np.array(y)

X = np.expand_dims(X, axis=-1)

print(f"Oblik ulaznih podataka (X): {X.shape}")
print(f"Oblik labela (y):           {y.shape}")

# Čuvanje gotovog dataset-a
np.savez_compressed("dataset_mel_spektrogrami.npz", X=X, y=y)
print("Sačuvano u 'dataset_mel_spektrogrami.npz'!")

Započeta ekstrakcija Mel-spektrograma za 1440 fajlova...
Obrađeno 200/1440...
Obrađeno 400/1440...
Obrađeno 600/1440...
Obrađeno 800/1440...
Obrađeno 1000/1440...
Obrađeno 1200/1440...
Obrađeno 1400/1440...
Obrađeno 1440/1440...
Oblik ulaznih podataka (X): (1440, 128, 110, 1)
Oblik labela (y):           (1440,)
Sačuvano u 'dataset_mel_spektrogrami.npz'!


#### resavanje problema manjka neutrala

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

def spec_augment(mel_spec, freq_mask_max=8, time_mask_max=12):
    augmented = mel_spec.copy()
    
    # Provera i prilagođavanje dimenzija ako spektrogram ima kanal na kraju
    is_3d = (augmented.ndim == 3 and augmented.shape[-1] == 1)
    if is_3d:
        augmented = augmented[:, :, 0]
        
    num_freqs, num_steps = augmented.shape
    
    # Frequency Masking
    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, num_freqs - f)
    augmented[f0:f0+f, :] = 0
    
    # Time Masking
    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, num_steps - t)
    augmented[:, t0:t0+t] = 0
    
    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)
        
    return augmented

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

data = np.load("dataset_mel_spektrogrami.npz")
X_all, y_all = data["X"], data["y"]

X_tr, y_tr = [], []
X_va, y_va = [], []
X_te, y_te = [], []

for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    if c == 0:  
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        # Izdvajanje originalnih neutralnih uzoraka
        X_orig = X_all[train_idx]
        
        # Generisanje modifikovanih uzoraka primenom SpecAugment-a
        X_aug = np.array([spec_augment(x) for x in X_orig])
        
        # Spajanje 64 originalna i 64 augmentisana uzorka u 128
        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])

    else:  
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])

X_train = np.concatenate(X_tr, axis=0)
y_train = np.concatenate(y_tr, axis=0)
train_perm = np.random.permutation(len(y_train))
X_train, y_train = X_train[train_perm], y_train[train_perm]

X_val = np.concatenate(X_va, axis=0)
y_val = np.concatenate(y_va, axis=0)
val_perm = np.random.permutation(len(y_val))
X_val, y_val = X_val[val_perm], y_val[val_perm]

X_test = np.concatenate(X_te, axis=0)
y_test = np.concatenate(y_te, axis=0)
test_perm = np.random.permutation(len(y_test))
X_test, y_test = X_test[test_perm], y_test[test_perm]

mean = np.mean(X_train, axis=2, keepdims=True)
std = np.std(X_train, axis=2, keepdims=True)
X_train = (X_train - mean) / (std + 1e-8)

mean_val = np.mean(X_val, axis=2, keepdims=True)
std_val = np.std(X_val, axis=2, keepdims=True)
X_val = (X_val - mean_val) / (std_val + 1e-8)

mean_test = np.mean(X_test, axis=2, keepdims=True)
std_test = np.std(X_test, axis=2, keepdims=True)
X_test = (X_test - mean_test) / (std_test + 1e-8)

print("--- RASPODELA UZORAKA (AUGMENTED NEUTRAL) ---")
print(f"Train skup: {len(X_train)} uzoraka (128 po klasi - neutral augmentovan SpecAugment-om)")
print(f"Val skup:   {len(X_val)} uzoraka (32 za ostale, 16 za neutral)")
print(f"Test skup:  {len(X_test)} uzoraka (32 za ostale, 16 za neutral)\n")


class AudioDataset(Dataset):
    def __init__(self, X, y):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader_logmel = DataLoader(AudioDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader_logmel = DataLoader(AudioDataset(X_val, y_val), batch_size=32, shuffle=False)
test_loader_logmel = DataLoader(AudioDataset(X_test, y_test), batch_size=32, shuffle=False)

Koristi se uređaj: cpu
--- RASPODELA UZORAKA (AUGMENTED NEUTRAL) ---
Train skup: 1024 uzoraka (128 po klasi - neutral augmentovan SpecAugment-om)
Val skup:   240 uzoraka (32 za ostale, 16 za neutral)
Test skup:  240 uzoraka (32 za ostale, 16 za neutral)



In [3]:
def normalize_per_coefficient(spec):
    """
    spec: matrica oblika (128, T) ili (128, T, 1)
    Normalizuje svaki od 128 Mel binarizovanih opsega nezavisno duž vremenske ose.
    """
    # Određujemo osu vremena (obično je axis=1)
    if spec.ndim == 3 and spec.shape[-1] == 1:
        time_axis = 1
    elif spec.ndim == 2:
        time_axis = 1
    else:
        time_axis = -1

    mean = np.mean(spec, axis=time_axis, keepdims=True)
    std = np.std(spec, axis=time_axis, keepdims=True)
    
    return (spec - mean) / (std + 1e-8)

#### 2N

In [4]:
import os
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# --- DEFINICIJA SPECAUGMENT FUNKCIJE ---
def spec_augment(mel_spec, freq_mask_max=8, time_mask_max=12):
    augmented = mel_spec.copy()
    
    is_3d = (augmented.ndim == 3 and augmented.shape[-1] == 1)
    if is_3d:
        augmented = augmented[:, :, 0]
        
    num_freqs, num_steps = augmented.shape
    
    # Frequency Masking
    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, max(1, num_freqs - f))
    augmented[f0:f0+f, :] = 0
    
    # Time Masking
    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, max(1, num_steps - t))
    augmented[:, t0:t0+t] = 0
    
    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)
        
    return augmented

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel2N"
os.makedirs(save_dir, exist_ok=True)

best_model_path = "model_logmel_2N.pth"

data = np.load("dataset_mel_spektrogrami.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0: 
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        # Augmentacija neutralne klase pomoću SpecAugment-a
        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment(x) for x in X_orig])
        
        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])
        

    else:  
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]


class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_logmel = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_logmel = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_logmel = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


class LOGMEL_2CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super(LOGMEL_2CNN_Small, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(2, 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(4, 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(8, 64),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

model = LOGMEL_2CNN_Small(num_classes=len(emotion_names), dropout_rate=0.3).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

epochs = 50
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    # --- VALIDACIJA ---
    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    # --- TESTIRANJE ---
    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )





Koristi se uređaj: cpu
Epoha 01/50 | Train Loss: 2.0911 | Val Loss: 2.0777 | Val Acc: 13.33% | Test Loss: 2.0777 | Test Acc: 12.50% [Model sačuvan -> model_logmel_2N.pth]
Epoha 02/50 | Train Loss: 2.0822 | Val Loss: 2.0734 | Val Acc: 17.92% | Test Loss: 2.0746 | Test Acc: 17.50% [Model sačuvan -> model_logmel_2N.pth]
Epoha 03/50 | Train Loss: 2.0832 | Val Loss: 2.0723 | Val Acc: 18.75% | Test Loss: 2.0741 | Test Acc: 20.00% [Model sačuvan -> model_logmel_2N.pth]
Epoha 04/50 | Train Loss: 2.0724 | Val Loss: 2.0713 | Val Acc: 18.75% | Test Loss: 2.0735 | Test Acc: 19.58% [Model sačuvan -> model_logmel_2N.pth]
Epoha 05/50 | Train Loss: 2.0650 | Val Loss: 2.0684 | Val Acc: 19.17% | Test Loss: 2.0717 | Test Acc: 19.58% [Model sačuvan -> model_logmel_2N.pth]
Epoha 06/50 | Train Loss: 2.0657 | Val Loss: 2.0656 | Val Acc: 20.83% | Test Loss: 2.0700 | Test Acc: 20.42% [Model sačuvan -> model_logmel_2N.pth]
Epoha 07/50 | Train Loss: 2.0611 | Val Loss: 2.0654 | Val Acc: 20.83% | Test Loss: 2.0701

In [5]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_2CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_logmel_2N.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

=== Predikcija za 67. fajl u test skupu ===
Stvarna emocija (Ground Truth): angry
Predviđena emocija:             calm (21.36%)
Vreme pojedinačne inferencije:  12.161 ms

Verovatnoće po klasama:
  neutral   :   9.02%
  calm      :  21.36%
  happy     :  10.90%
  sad       :  15.47%
  angry     :  15.36%
  fearful   :   7.48%
  disgust   :  18.73%
  surprised :   1.68%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 240
Ukupno trajanje:           0.1581 s
Prosečno vreme po uzorku:  0.659 ms
Brzina obrade (throughput): 1518.04 FPS (uzoraka`/s)


#### 4N

In [6]:
import os
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


def spec_augment(mel_spec, freq_mask_max=8, time_mask_max=12):
    augmented = mel_spec.copy()
    
    is_3d = (augmented.ndim == 3 and augmented.shape[-1] == 1)
    if is_3d:
        augmented = augmented[:, :, 0]
        
    num_freqs, num_steps = augmented.shape
    
    # Frequency Masking
    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, max(1, num_freqs - f))
    augmented[f0:f0+f, :] = 0
    
    # Time Masking
    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, max(1, num_steps - t))
    augmented[:, t0:t0+t] = 0
    
    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)
        
    return augmented

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel4N"
os.makedirs(save_dir, exist_ok=True)

best_model_path = "model_logmel_4N.pth"

data = np.load("dataset_mel_spektrogrami.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0: 
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        # Augmentacija neutralne klase pomoću SpecAugment-a
        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment(x) for x in X_orig])
        
        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])
        

    else:  
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]


class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_logmel = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_logmel = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_logmel = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


class LOGMEL_4CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super(LOGMEL_4CNN_Small, self).__init__()
        n=4
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 > 64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

model = LOGMEL_4CNN_Small(num_classes=len(emotion_names), dropout_rate=0.3).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

epochs = 50
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    # --- VALIDACIJA ---
    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    # --- TESTIRANJE ---
    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )





Koristi se uređaj: cpu
Epoha 01/50 | Train Loss: 2.0942 | Val Loss: 2.0952 | Val Acc: 13.33% | Test Loss: 2.0962 | Test Acc: 13.33% [Model sačuvan -> model_logmel_4N.pth]
Epoha 02/50 | Train Loss: 2.0861 | Val Loss: 2.0859 | Val Acc: 13.33% | Test Loss: 2.0887 | Test Acc: 13.33% [Model sačuvan -> model_logmel_4N.pth]
Epoha 03/50 | Train Loss: 2.0727 | Val Loss: 2.0769 | Val Acc: 13.33% | Test Loss: 2.0818 | Test Acc: 13.33% [Model sačuvan -> model_logmel_4N.pth]
Epoha 04/50 | Train Loss: 2.0657 | Val Loss: 2.0705 | Val Acc: 13.75% | Test Loss: 2.0767 | Test Acc: 13.33% [Model sačuvan -> model_logmel_4N.pth]
Epoha 05/50 | Train Loss: 2.0543 | Val Loss: 2.0639 | Val Acc: 13.33% | Test Loss: 2.0708 | Test Acc: 13.75% [Model sačuvan -> model_logmel_4N.pth]
Epoha 06/50 | Train Loss: 2.0413 | Val Loss: 2.0567 | Val Acc: 15.00% | Test Loss: 2.0649 | Test Acc: 17.50% [Model sačuvan -> model_logmel_4N.pth]
Epoha 07/50 | Train Loss: 2.0389 | Val Loss: 2.0508 | Val Acc: 23.75% | Test Loss: 2.0589

In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_4CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_logmel_4N.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

#### 8N

In [7]:
import os
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


def spec_augment(mel_spec, freq_mask_max=8, time_mask_max=12):
    augmented = mel_spec.copy()
    
    is_3d = (augmented.ndim == 3 and augmented.shape[-1] == 1)
    if is_3d:
        augmented = augmented[:, :, 0]
        
    num_freqs, num_steps = augmented.shape
    
    # Frequency Masking
    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, max(1, num_freqs - f))
    augmented[f0:f0+f, :] = 0
    
    # Time Masking
    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, max(1, num_steps - t))
    augmented[:, t0:t0+t] = 0
    
    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)
        
    return augmented

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel8N"
os.makedirs(save_dir, exist_ok=True)

best_model_path = "model_logmel_8N.pth"

data = np.load("dataset_mel_spektrogrami.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0: 
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        # Augmentacija neutralne klase pomoću SpecAugment-a
        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment(x) for x in X_orig])
        
        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])
        

    else:  
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]


class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_logmel = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_logmel = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_logmel = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


class LOGMEL_8CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super(LOGMEL_8CNN_Small, self).__init__()
        n=8
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 > 64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

model = LOGMEL_8CNN_Small(num_classes=len(emotion_names), dropout_rate=0.3).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

epochs = 50
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    # --- VALIDACIJA ---
    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    # --- TESTIRANJE ---
    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )






Koristi se uređaj: cpu
Epoha 01/50 | Train Loss: 2.0738 | Val Loss: 2.0793 | Val Acc: 9.58% | Test Loss: 2.0756 | Test Acc: 11.67% [Model sačuvan -> model_logmel_8N.pth]
Epoha 02/50 | Train Loss: 2.0531 | Val Loss: 2.0760 | Val Acc: 10.83% | Test Loss: 2.0698 | Test Acc: 12.50% [Model sačuvan -> model_logmel_8N.pth]
Epoha 03/50 | Train Loss: 2.0333 | Val Loss: 2.0757 | Val Acc: 7.50% | Test Loss: 2.0696 | Test Acc: 10.83% [Model sačuvan -> model_logmel_8N.pth]
Epoha 04/50 | Train Loss: 2.0160 | Val Loss: 2.0767 | Val Acc: 7.92% | Test Loss: 2.0703 | Test Acc: 10.00%
Epoha 05/50 | Train Loss: 1.9824 | Val Loss: 2.0611 | Val Acc: 17.92% | Test Loss: 2.0534 | Test Acc: 18.75% [Model sačuvan -> model_logmel_8N.pth]
Epoha 06/50 | Train Loss: 1.9626 | Val Loss: 2.0336 | Val Acc: 25.42% | Test Loss: 2.0265 | Test Acc: 27.50% [Model sačuvan -> model_logmel_8N.pth]
Epoha 07/50 | Train Loss: 1.9394 | Val Loss: 2.0326 | Val Acc: 24.58% | Test Loss: 2.0268 | Test Acc: 23.75% [Model sačuvan -> mode

In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_8CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_logmel_8N.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

#### 16N

In [8]:
import os
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


def spec_augment(mel_spec, freq_mask_max=8, time_mask_max=12):
    augmented = mel_spec.copy()
    
    is_3d = (augmented.ndim == 3 and augmented.shape[-1] == 1)
    if is_3d:
        augmented = augmented[:, :, 0]
        
    num_freqs, num_steps = augmented.shape
    
    # Frequency Masking
    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, max(1, num_freqs - f))
    augmented[f0:f0+f, :] = 0
    
    # Time Masking
    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, max(1, num_steps - t))
    augmented[:, t0:t0+t] = 0
    
    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)
        
    return augmented

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel16N"
os.makedirs(save_dir, exist_ok=True)

best_model_path = "model_logmel_16N.pth"

data = np.load("dataset_mel_spektrogrami.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0: 
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        # Augmentacija neutralne klase pomoću SpecAugment-a
        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment(x) for x in X_orig])
        
        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])
        

    else:  
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]


class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_logmel = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_logmel = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_logmel = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


class LOGMEL_16CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super(LOGMEL_16CNN_Small, self).__init__()
        n=16
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 > 64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

model = LOGMEL_16CNN_Small(num_classes=len(emotion_names), dropout_rate=0.3).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

epochs = 50
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    # --- VALIDACIJA ---
    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    # --- TESTIRANJE ---
    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )





Koristi se uređaj: cpu
Epoha 01/50 | Train Loss: 2.0640 | Val Loss: 2.0752 | Val Acc: 6.67% | Test Loss: 2.0711 | Test Acc: 8.33% [Model sačuvan -> model_logmel_16N.pth]
Epoha 02/50 | Train Loss: 2.0080 | Val Loss: 2.0546 | Val Acc: 13.33% | Test Loss: 2.0506 | Test Acc: 15.00% [Model sačuvan -> model_logmel_16N.pth]
Epoha 03/50 | Train Loss: 1.9653 | Val Loss: 2.0350 | Val Acc: 15.42% | Test Loss: 2.0326 | Test Acc: 15.42% [Model sačuvan -> model_logmel_16N.pth]
Epoha 04/50 | Train Loss: 1.9155 | Val Loss: 1.9803 | Val Acc: 27.50% | Test Loss: 1.9836 | Test Acc: 26.67% [Model sačuvan -> model_logmel_16N.pth]
Epoha 05/50 | Train Loss: 1.8640 | Val Loss: 1.9412 | Val Acc: 22.08% | Test Loss: 1.9512 | Test Acc: 22.50% [Model sačuvan -> model_logmel_16N.pth]
Epoha 06/50 | Train Loss: 1.8279 | Val Loss: 1.8814 | Val Acc: 32.08% | Test Loss: 1.8943 | Test Acc: 32.08% [Model sačuvan -> model_logmel_16N.pth]
Epoha 07/50 | Train Loss: 1.7683 | Val Loss: 1.8146 | Val Acc: 35.42% | Test Loss: 1.

In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_16CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_logmel_16N.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

#### 32N

In [9]:
import os
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


def spec_augment(mel_spec, freq_mask_max=8, time_mask_max=12):
    augmented = mel_spec.copy()
    
    is_3d = (augmented.ndim == 3 and augmented.shape[-1] == 1)
    if is_3d:
        augmented = augmented[:, :, 0]
        
    num_freqs, num_steps = augmented.shape
    
    # Frequency Masking
    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, max(1, num_freqs - f))
    augmented[f0:f0+f, :] = 0
    
    # Time Masking
    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, max(1, num_steps - t))
    augmented[:, t0:t0+t] = 0
    
    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)
        
    return augmented

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel32N"
os.makedirs(save_dir, exist_ok=True)

best_model_path = "model_logmel_32N.pth"

data = np.load("dataset_mel_spektrogrami.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0: 
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        # Augmentacija neutralne klase pomoću SpecAugment-a
        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment(x) for x in X_orig])
        
        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])
        

    else:  
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]


class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_logmel = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_logmel = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_logmel = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


class LOGMEL_32CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super(LOGMEL_32CNN_Small, self).__init__()
        n=32
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 > 64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

model = LOGMEL_32CNN_Small(num_classes=len(emotion_names), dropout_rate=0.3).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

epochs = 50
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    # --- VALIDACIJA ---
    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    # --- TESTIRANJE ---
    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )





Koristi se uređaj: cpu
Epoha 01/50 | Train Loss: 2.0146 | Val Loss: 2.2763 | Val Acc: 6.67% | Test Loss: 2.2681 | Test Acc: 6.67% [Model sačuvan -> model_logmel_32N.pth]
Epoha 02/50 | Train Loss: 1.9441 | Val Loss: 2.0113 | Val Acc: 28.33% | Test Loss: 2.0133 | Test Acc: 23.33% [Model sačuvan -> model_logmel_32N.pth]
Epoha 03/50 | Train Loss: 1.8811 | Val Loss: 1.9548 | Val Acc: 30.00% | Test Loss: 1.9611 | Test Acc: 25.83% [Model sačuvan -> model_logmel_32N.pth]
Epoha 04/50 | Train Loss: 1.8341 | Val Loss: 1.9073 | Val Acc: 34.58% | Test Loss: 1.9151 | Test Acc: 32.50% [Model sačuvan -> model_logmel_32N.pth]
Epoha 05/50 | Train Loss: 1.7928 | Val Loss: 1.8339 | Val Acc: 33.75% | Test Loss: 1.8484 | Test Acc: 32.92% [Model sačuvan -> model_logmel_32N.pth]
Epoha 06/50 | Train Loss: 1.7207 | Val Loss: 1.7950 | Val Acc: 33.75% | Test Loss: 1.8081 | Test Acc: 32.50% [Model sačuvan -> model_logmel_32N.pth]
Epoha 07/50 | Train Loss: 1.6687 | Val Loss: 1.7331 | Val Acc: 34.17% | Test Loss: 1.

In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_32CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_logmel_32N.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

#### 64N

In [10]:
import os
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


def spec_augment(mel_spec, freq_mask_max=8, time_mask_max=12):
    augmented = mel_spec.copy()
    
    is_3d = (augmented.ndim == 3 and augmented.shape[-1] == 1)
    if is_3d:
        augmented = augmented[:, :, 0]
        
    num_freqs, num_steps = augmented.shape
    
    # Frequency Masking
    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, max(1, num_freqs - f))
    augmented[f0:f0+f, :] = 0
    
    # Time Masking
    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, max(1, num_steps - t))
    augmented[:, t0:t0+t] = 0
    
    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)
        
    return augmented

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel64N"
os.makedirs(save_dir, exist_ok=True)

best_model_path = "model_logmel_64N.pth"

data = np.load("dataset_mel_spektrogrami.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0: 
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        # Augmentacija neutralne klase pomoću SpecAugment-a
        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment(x) for x in X_orig])
        
        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])
        

    else:  
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]


class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_logmel = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_logmel = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_logmel = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


class LOGMEL_64CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super(LOGMEL_64CNN_Small, self).__init__()
        n=64
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 > 64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

model = LOGMEL_64CNN_Small(num_classes=len(emotion_names), dropout_rate=0.3).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

epochs = 50
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    # --- VALIDACIJA ---
    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    # --- TESTIRANJE ---
    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )





Koristi se uređaj: cpu
Epoha 01/50 | Train Loss: 2.0032 | Val Loss: 2.1841 | Val Acc: 6.67% | Test Loss: 2.1780 | Test Acc: 7.08% [Model sačuvan -> model_logmel_64N.pth]
Epoha 02/50 | Train Loss: 1.8970 | Val Loss: 1.9565 | Val Acc: 29.17% | Test Loss: 1.9608 | Test Acc: 26.25% [Model sačuvan -> model_logmel_64N.pth]
Epoha 03/50 | Train Loss: 1.8211 | Val Loss: 1.9028 | Val Acc: 30.83% | Test Loss: 1.9062 | Test Acc: 28.33% [Model sačuvan -> model_logmel_64N.pth]
Epoha 04/50 | Train Loss: 1.7634 | Val Loss: 1.8728 | Val Acc: 27.50% | Test Loss: 1.8692 | Test Acc: 27.50% [Model sačuvan -> model_logmel_64N.pth]
Epoha 05/50 | Train Loss: 1.6832 | Val Loss: 1.8221 | Val Acc: 28.75% | Test Loss: 1.8108 | Test Acc: 30.00% [Model sačuvan -> model_logmel_64N.pth]
Epoha 06/50 | Train Loss: 1.6248 | Val Loss: 1.7036 | Val Acc: 38.33% | Test Loss: 1.7042 | Test Acc: 35.00% [Model sačuvan -> model_logmel_64N.pth]
Epoha 07/50 | Train Loss: 1.5861 | Val Loss: 1.8564 | Val Acc: 25.42% | Test Loss: 1.

In [11]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_64CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_logmel_64N.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

=== Predikcija za 67. fajl u test skupu ===
Stvarna emocija (Ground Truth): angry
Predviđena emocija:             angry (66.42%)
Vreme pojedinačne inferencije:  42.590 ms

Verovatnoće po klasama:
  neutral   :   0.11%
  calm      :   0.02%
  happy     :   3.00%
  sad       :   3.40%
  angry     :  66.42%
  fearful   :   0.63%
  disgust   :  26.28%
  surprised :   0.13%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 240
Ukupno trajanje:           4.5912 s
Prosečno vreme po uzorku:  19.130 ms
Brzina obrade (throughput): 52.27 FPS (uzoraka`/s)


#### 128N

In [13]:
import os
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


def spec_augment(mel_spec, freq_mask_max=8, time_mask_max=12):
    augmented = mel_spec.copy()
    
    is_3d = (augmented.ndim == 3 and augmented.shape[-1] == 1)
    if is_3d:
        augmented = augmented[:, :, 0]
        
    num_freqs, num_steps = augmented.shape
    
    # Frequency Masking
    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, max(1, num_freqs - f))
    augmented[f0:f0+f, :] = 0
    
    # Time Masking
    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, max(1, num_steps - t))
    augmented[:, t0:t0+t] = 0
    
    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)
        
    return augmented

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel128N"
os.makedirs(save_dir, exist_ok=True)

best_model_path = "model_logmel_128N.pth"

data = np.load("dataset_mel_spektrogrami.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0: 
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        # Augmentacija neutralne klase pomoću SpecAugment-a
        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment(x) for x in X_orig])
        
        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])
        

    else:  
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]


class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_logmel = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_logmel = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_logmel = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


class LOGMEL_128CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super(LOGMEL_128CNN_Small, self).__init__()
        n=128
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 > 64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

model = LOGMEL_128CNN_Small(num_classes=len(emotion_names), dropout_rate=0.3).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

epochs = 50
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    # --- VALIDACIJA ---
    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    # --- TESTIRANJE ---
    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )





Koristi se uređaj: cpu
Epoha 01/50 | Train Loss: 1.9631 | Val Loss: 2.4426 | Val Acc: 6.67% | Test Loss: 2.4588 | Test Acc: 7.50% [Model sačuvan -> model_logmel_128N.pth]
Epoha 02/50 | Train Loss: 1.7949 | Val Loss: 1.8340 | Val Acc: 30.42% | Test Loss: 1.8544 | Test Acc: 35.00% [Model sačuvan -> model_logmel_128N.pth]
Epoha 03/50 | Train Loss: 1.6991 | Val Loss: 1.7308 | Val Acc: 33.33% | Test Loss: 1.7524 | Test Acc: 35.42% [Model sačuvan -> model_logmel_128N.pth]
Epoha 04/50 | Train Loss: 1.6374 | Val Loss: 1.9519 | Val Acc: 23.33% | Test Loss: 1.9385 | Test Acc: 22.92%
Epoha 05/50 | Train Loss: 1.5995 | Val Loss: 1.6526 | Val Acc: 36.67% | Test Loss: 1.7001 | Test Acc: 35.00% [Model sačuvan -> model_logmel_128N.pth]
Epoha 06/50 | Train Loss: 1.5415 | Val Loss: 1.7177 | Val Acc: 34.17% | Test Loss: 1.7045 | Test Acc: 33.33%
Epoha 07/50 | Train Loss: 1.5073 | Val Loss: 1.5997 | Val Acc: 38.33% | Test Loss: 1.6224 | Test Acc: 38.33% [Model sačuvan -> model_logmel_128N.pth]
Epoha 08/50

In [17]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_128CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_logmel_128N.pth", map_location=device))
model.eval()

sample_idx = 64
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

=== Predikcija za 65. fajl u test skupu ===
Stvarna emocija (Ground Truth): sad
Predviđena emocija:             neutral (24.43%)
Vreme pojedinačne inferencije:  123.561 ms

Verovatnoće po klasama:
  neutral   :  24.43%
  calm      :  19.47%
  happy     :  12.90%
  sad       :  22.55%
  angry     :   5.93%
  fearful   :   6.01%
  disgust   :   6.91%
  surprised :   1.80%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 240
Ukupno trajanje:           12.9901 s
Prosečno vreme po uzorku:  54.125 ms
Brzina obrade (throughput): 18.48 FPS (uzoraka`/s)


#### 256N

In [ ]:
import os
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


def spec_augment(mel_spec, freq_mask_max=8, time_mask_max=12):
    augmented = mel_spec.copy()
    
    is_3d = (augmented.ndim == 3 and augmented.shape[-1] == 1)
    if is_3d:
        augmented = augmented[:, :, 0]
        
    num_freqs, num_steps = augmented.shape
    
    # Frequency Masking
    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, max(1, num_freqs - f))
    augmented[f0:f0+f, :] = 0
    
    # Time Masking
    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, max(1, num_steps - t))
    augmented[:, t0:t0+t] = 0
    
    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)
        
    return augmented

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel256N"
os.makedirs(save_dir, exist_ok=True)

best_model_path = "model_logmel_256N.pth"

data = np.load("dataset_mel_spektrogrami.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0: 
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        # Augmentacija neutralne klase pomoću SpecAugment-a
        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment(x) for x in X_orig])
        
        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])
        

    else:  
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]


class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_logmel = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_logmel = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_logmel = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


class LOGMEL_256CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super(LOGMEL_256CNN_Small, self).__init__()
        n=256
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 > 64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

model = LOGMEL_256CNN_Small(num_classes=len(emotion_names), dropout_rate=0.3).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

epochs = 50
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    # --- VALIDACIJA ---
    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    # --- TESTIRANJE ---
    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )





Koristi se uređaj: cpu


In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_256CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_logmel_256N.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

#### 512N

In [ ]:
import os
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


def spec_augment(mel_spec, freq_mask_max=8, time_mask_max=12):
    augmented = mel_spec.copy()
    
    is_3d = (augmented.ndim == 3 and augmented.shape[-1] == 1)
    if is_3d:
        augmented = augmented[:, :, 0]
        
    num_freqs, num_steps = augmented.shape
    
    # Frequency Masking
    f = np.random.randint(1, freq_mask_max)
    f0 = np.random.randint(0, max(1, num_freqs - f))
    augmented[f0:f0+f, :] = 0
    
    # Time Masking
    t = np.random.randint(1, time_mask_max)
    t0 = np.random.randint(0, max(1, num_steps - t))
    augmented[:, t0:t0+t] = 0
    
    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)
        
    return augmented

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel512N"
os.makedirs(save_dir, exist_ok=True)

best_model_path = "model_logmel_512N.pth"

data = np.load("dataset_mel_spektrogrami.npz")
X_all, y_all = data["X"], data["y"]
filenames_all = data["filenames"] if "filenames" in data else None

X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

for c in range(8):
    indices = np.where(y_all == c)[0]
    np.random.shuffle(indices)

    weights_c = []
    for idx in indices:
        if filenames_all is not None:
            fname = os.path.basename(str(filenames_all[idx]))
            parts = fname.split("-")
            if len(parts) >= 4 and parts[3] == "02":
                weights_c.append(2.0)
            else:
                weights_c.append(1.0)
        else:
            weights_c.append(1.0)
    weights_c = np.array(weights_c, dtype=np.float32)

    if c == 0: 
        train_idx = indices[:64]
        val_idx = indices[64:80]
        test_idx = indices[80:96]

        train_w = weights_c[:64]

        # Augmentacija neutralne klase pomoću SpecAugment-a
        X_orig = X_all[train_idx]
        X_aug = np.array([spec_augment(x) for x in X_orig])
        
        X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
        y_train_neutral = np.tile(y_all[train_idx], 2)
        w_train_neutral = np.tile(train_w, 2)

        X_tr.append(X_train_neutral)
        y_tr.append(y_train_neutral)
        w_tr.append(w_train_neutral)

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[64:80])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[80:96])
        

    else:  
        train_idx = indices[:128]
        val_idx = indices[128:160]
        test_idx = indices[160:192]

        X_tr.append(X_all[train_idx])
        y_tr.append(y_all[train_idx])
        w_tr.append(weights_c[:128])

        X_va.append(X_all[val_idx])
        y_va.append(y_all[val_idx])
        w_va.append(weights_c[128:160])

        X_te.append(X_all[test_idx])
        y_te.append(y_all[test_idx])
        w_te.append(weights_c[160:192])

X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]


class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_logmel = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
)
val_loader_logmel = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
)
test_loader_logmel = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
)


class LOGMEL_512CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super(LOGMEL_512CNN_Small, self).__init__()
        n=512
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        if n*4 > 64:
            k=64
        else:
            k=n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

model = LOGMEL_512CNN_Small(num_classes=len(emotion_names), dropout_rate=0.3).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

epochs = 50
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    # --- VALIDACIJA ---
    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), best_model_path)
        saved_flag = f" [Model sačuvan -> {best_model_path}]"

    # --- TESTIRANJE ---
    test_y_true, test_y_pred = [], []
    running_test_loss = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in test_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_test_loss += loss.item() * inputs.size(0)
            total_test_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            test_y_true.extend(labels.cpu().numpy())
            test_y_pred.extend(preds)

    epoch_test_loss = running_test_loss / total_test_samples
    cm_test, _, test_acc = compute_epoch_metrics(
        test_y_true, test_y_pred, emotion_names
    )

    csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
    df_metrics.to_csv(csv_path, index=False)

    summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(f"Epoha: {epoch}\n")
        f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
        f.write(
            f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
        )
        f.write(
            f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
        )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_val,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=emotion_names,
        yticklabels=emotion_names,
    )
    plt.xlabel("Predviđena emocija")
    plt.ylabel("Stvarna emocija")
    plt.title(
        f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
    )
    plt.tight_layout()

    cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
    )





In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_512CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_logmel_512N.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")